# Pricing Environment

Gymnasium-compatible simulator for the dynamic pricing RL problem.

**State**: 11-dim normalized feature vector (ride request context)  
**Actions**: 5 discrete price multipliers → `[0.8x, 1.0x, 1.2x, 1.5x, 2.0x]`  
**Reward**: `adjusted_price` if rider accepts, else `-0.1 × base_cost` (opportunity cost)

Demand elasticity is simulated from dataset features:
- High demand/supply ratio → riders tolerate higher prices
- Gold loyalty → less price-sensitive
- Premium vehicle → slightly captive demand

In [1]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

PRICE_MULTIPLIERS = np.array([0.8, 1.0, 1.2, 1.5, 2.0], dtype=np.float32)
N_ACTIONS = len(PRICE_MULTIPLIERS)

# Indices within the scaled state vector (matches STATE_COLS in preprocessing notebook)
IDX_RIDERS   = 0
IDX_DRIVERS  = 1
IDX_DSR      = 2   # demand_supply_ratio
IDX_IS_SURGE = 3
IDX_LOCATION = 4
IDX_LOYALTY  = 5
IDX_PAST_RIDES = 6
IDX_RATINGS  = 7
IDX_TIME     = 8
IDX_VEHICLE  = 9
IDX_DURATION = 10

print(f"Actions : {N_ACTIONS}  →  multipliers {PRICE_MULTIPLIERS}")

Actions : 5  →  multipliers [0.8 1.  1.2 1.5 2. ]


In [2]:
class PricingEnv(gym.Env):
    """
    Episode: iterate over dataset rows (shuffled each reset).
    Each step = one ride request; agent picks a price multiplier.
    """
    metadata = {"render_modes": []}

    def __init__(self, X: np.ndarray, y: np.ndarray, seed: int = 42):
        super().__init__()
        self.X = X.astype(np.float32)
        self.y = y.astype(np.float32)
        self.n_samples = len(X)
        self._rng = np.random.default_rng(seed)

        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(X.shape[1],), dtype=np.float32
        )
        self.action_space = spaces.Discrete(N_ACTIONS)
        self._order = np.arange(self.n_samples)
        self._cursor = 0

    def _demand_factor(self, state: np.ndarray, multiplier: float) -> float:
        """
        Acceptance probability model:
          - Base acceptance at 1.0x ≈ 0.85
          - Elasticity reduced by high demand, loyalty, premium vehicle
        """
        dsr     = float(state[IDX_DSR])
        loyalty = float(state[IDX_LOYALTY])
        vehicle = float(state[IDX_VEHICLE])

        base_elasticity = 0.35
        elasticity = base_elasticity * (1.0 - 0.4 * dsr) * (1.0 - 0.2 * loyalty) * (1.0 - 0.1 * vehicle)

        acceptance = 0.85 - elasticity * (multiplier - 1.0)
        return float(np.clip(acceptance, 0.0, 1.0))

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._order = self._rng.permutation(self.n_samples)
        self._cursor = 0
        return self.X[self._order[self._cursor]], {}

    def step(self, action: int):
        idx        = self._order[self._cursor]
        state      = self.X[idx]
        base_cost  = float(self.y[idx])
        multiplier = float(PRICE_MULTIPLIERS[action])

        adjusted_price = base_cost * multiplier
        demand_prob    = self._demand_factor(state, multiplier)
        accepted       = self._rng.random() < demand_prob

        reward = adjusted_price if accepted else -base_cost * 0.1

        self._cursor += 1
        terminated = self._cursor >= self.n_samples
        next_obs   = (
            self.X[self._order[self._cursor]]
            if not terminated
            else np.zeros(self.observation_space.shape, dtype=np.float32)
        )

        info = {
            "base_cost":      base_cost,
            "multiplier":     multiplier,
            "adjusted_price": adjusted_price,
            "accepted":       accepted,
            "demand_prob":    demand_prob,
        }
        return next_obs, float(reward), terminated, False, info

    def render(self):
        pass

print("PricingEnv defined.")

PricingEnv defined.


## Smoke test — random policy

In [3]:
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data/processed")

X = np.load(os.path.join(PROCESSED_DIR, "X.npy"))
y = np.load(os.path.join(PROCESSED_DIR, "y.npy"))

env = PricingEnv(X, y, seed=42)
print(f"Obs space : {env.observation_space}")
print(f"Act space : {env.action_space}")

obs, _ = env.reset()
total_rev, n_accepted = 0, 0
done = False
while not done:
    a = env.action_space.sample()
    obs, r, terminated, truncated, info = env.step(a)
    done = terminated or truncated
    if info["accepted"]:
        total_rev  += info["adjusted_price"]
        n_accepted += 1

print(f"\nRandom policy — Revenue: ${total_rev:,.1f} | Accepted: {n_accepted}/{env.n_samples} ({n_accepted/env.n_samples:.1%})")

Obs space : Box(0.0, 1.0, (11,), float32)
Act space : Discrete(5)

Random policy — Revenue: $348,686.5 | Accepted: 756/1000 (75.6%)
